# 02 - Ingest + silver

Task `Step_01_Ingest_Silver` of the pipeline job. Three stages, each logged to `ops.pipeline_runs` under its own task name (`load_user_data`, `bronze_ingest`, `silver_transform`; the last two unchanged, so the processed-run bookkeeping and the bronze retention in `04_gold_cleanup` keep working).

**0. App user data** - `config/user_data.json` in the pipeline volume (Supabase export copied by `databricks/jobs/sync_user_data.py` in GitHub Actions) -> `ops.user_profile`, `gold.user_job_state`, `ops.custom_roles`, `ops.custom_skills`, when the file is newer than the last load. A missing or invalid file keeps the tables as they are and never stops the run (details in section 0).

**1. Bronze** - new scraper CSVs from the landing volume -> `bronze.jobs_raw` with **Auto Loader** (`availableNow`)
* Auto Loader keeps processed files in its checkpoint (`ops.pipeline/checkpoints/bronze_jobs_raw`), so every file is ingested exactly once, even if the job is triggered several times.
* All business columns are kept as `STRING` exactly as in the file; typing and cleaning happen in silver. The LinkedIn criteria columns are trailing, so older files simply leave them `NULL`.
* Values that don't fit the schema go to `_rescued_data` instead of failing the run.
* Debug columns: `_source_file`, `_source_system`, `_landing_folder_date`, `_file_modification_time`, `_file_size`, `_row_hash`, `_ingested_at`, `_job_run_id`.

**2. Silver** - bronze rows whose `_job_run_id` has not yet been handled by a successful silver run (tracked in `ops.pipeline_runs.details.bronze_run_ids`; a failed run is picked up next time) are cleaned and MERGEd into `silver.jobs_clean` (one row per `job_key`)

| Step | Rule |
|---|---|
| Trim | All strings trimmed, empty -> `NULL`, whitespace in descriptions collapsed |
| Reject | Missing `job_id`, `title`, `ats_type` or `job_url` -> `silver.jobs_rejected` with reason |
| Company | Missing -> `Unknown` (old LinkedIn files: `LinkedIn \| ` prefix removed) |
| LinkedIn ids | Legacy slug ids (`title-at-company-4012345678`) -> numeric id, tracking params removed from URL |
| Key | `job_key = sha2(lower(ats_type) \| lower(company_name) \| job_id)` |
| Dates | `dd/MM/yyyy` -> `DATE`, unparseable (e.g. `Posted 30+ Days Ago`) -> `NULL` |
| LinkedIn criteria | CSV columns `seniority_level`, `employment_type`, `job_function`, `industries` (old files: parsed from the trailing block of the description) |
| Location | `is_india` via `ops.ref_india_locations` (handles `Gurgaon`, `Bangalore`...), `country` = last part of the first location. Non-India jobs are counted, not stored |
| Dedupe | Latest sighting with a description wins per `job_key`; `first_seen_at`, `last_seen_at`, `times_seen` maintained across runs |
| Re-sightings | An empty description, criteria value, location or posted date never overwrites a stored one (LinkedIn re-emits already seen jobs without fetching their description) |

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
import re

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "silver", "all"],
                         "Full refresh (silver: reprocess all bronze; all: also re-ingest all files)")

LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
FULL_REFRESH = dbutils.widgets.get("full_refresh")

if not LANDING_PATH.startswith("/Volumes/"):
    raise ValueError(f"landing_path must be a Unity Catalog volume path: {LANDING_PATH}")

BRONZE_TABLE = table("bronze", "jobs_raw")
SILVER_TABLE = table("silver", "jobs_clean")
REJECTED_TABLE = table("silver", "jobs_rejected")
RUNS_TABLE = table("ops", "pipeline_runs")
INDIA_REF_TABLE = table("ops", "ref_india_locations")
CHECKPOINT = f"{PIPELINE_VOLUME}/checkpoints/bronze_jobs_raw"

# scraper_common.OUTPUT_COLUMNS order; new CSV columns are only ever appended
CRITERIA_COLUMNS = ["seniority_level", "employment_type", "job_function", "industries"]
BUSINESS_COLUMNS = [
    "company_name", "ats_type", "company_url", "job_id", "title", "location",
    "posted_date", "job_url", "description", "collected_on", "posted_days", *CRITERIA_COLUMNS,
]
CSV_SCHEMA = ", ".join(f"{column} STRING" for column in BUSINESS_COLUMNS)

SILVER_COLUMNS = [
    "job_key", "source", "ats_type", "company_name", "company_url", "job_id", "title", "location", "country", "is_india",
    "posted_date", "posted_days", "collected_on", "job_url", "description", "description_raw_length", "description_hash",
    *CRITERIA_COLUMNS, "dq_missing_description",
    "first_seen_at", "last_seen_at", "times_seen", "_source_file", "_row_hash", "_ingested_at", "_job_run_id", "_updated_at",
]
REJECTED_COLUMNS = [
    "reject_reason", "company_name", "ats_type", "job_id", "title", "job_url", "location",
    "_rescued_data", "_source_file", "_row_hash", "_ingested_at", "_job_run_id", "_rejected_at",
]

print(f"landing={LANDING_PATH}\ncheckpoint={CHECKPOINT}\nbronze={BRONZE_TABLE} silver={SILVER_TABLE}\nfull_refresh={FULL_REFRESH}")

## Optional full refresh
Only for development. `silver` empties the silver tables so every bronze run is processed again; `all` also empties bronze and resets the checkpoint so every file in the volume is ingested again. Silver dedupes by `job_key`, so re-ingesting is safe but slower.

In [ ]:
if FULL_REFRESH == "all":
    spark.sql(f"TRUNCATE TABLE {BRONZE_TABLE}")
    dbutils.fs.rm(CHECKPOINT, recurse=True)
    print("Bronze truncated and checkpoint removed")
if FULL_REFRESH in ("silver", "all"):
    spark.sql(f"TRUNCATE TABLE {SILVER_TABLE}")
    spark.sql(f"TRUNCATE TABLE {REJECTED_TABLE}")
    print("Silver tables truncated; all bronze runs will be reprocessed")

## 0. App user data
`config/user_data.json` -> `ops.user_profile`, `gold.user_job_state`, `ops.custom_roles`, `ops.custom_skills` (task `load_user_data`). The app keeps user data in Supabase; `databricks/jobs/sync_user_data.py` (GitHub Actions, before it starts this job) copies it into the volume, so Databricks never holds a Supabase credential.
* Missing, unreadable or invalid file -> the tables are kept and the run carries on (`details.loaded = false`, `skipped_reason`). A load error is logged as FAILED and the run carries on too; the next run loads the file again.
* Skipped when `exported_at` is not newer than the last successful load, and when the export has no profiles while `ops.user_profile` has some (`empty export`: an empty or failed export never wipes user data).
* `ops.user_profile` and `gold.user_job_state` are replaced (INSERT OVERWRITE, every column; the export only carries tracked job state - applied or a status - which is all the pipeline uses). `ops.custom_roles` is merged on `lower(role_title)` (the newer `checked_at` wins, `attempts` = the greater, never deleted) and `ops.custom_skills` gets the missing skills. `03_enrich` stays the authority for custom role checks; `04_gold_cleanup` publishes them back.

In [ ]:
import time
from datetime import timedelta

USER_DATA_FILE = f"{PIPELINE_VOLUME}/config/user_data.json"
USER_DATA_FORMAT = "jobseeker.user_data.v1"
PROFILE_TABLE = table("ops", "user_profile")
USER_STATE_TABLE = table("gold", "user_job_state")
CUSTOM_ROLES_TABLE = table("ops", "custom_roles")
CUSTOM_SKILLS_TABLE = table("ops", "custom_skills")

UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
JOB_KEY_RE = re.compile(r"[0-9a-f]{64}")
ISO_TS_RE = re.compile(r"(\d{4}-\d{2}-\d{2})[T ](\d{2}:\d{2}:\d{2})(?:\.(\d{1,9}))?(Z|[+-]\d{2}:?\d{2})?")
CUSTOM_ROLE_STATUSES = {"pending", "active", "mapped", "rejected"}

PROFILE_SCHEMA = ("profile_id STRING, email STRING, target_roles ARRAY<STRING>, skills ARRAY<STRING>, min_years INT, "
                  "max_years INT, preferred_cities ARRAY<STRING>, updated_at STRING")
STATE_SCHEMA = ("user_id STRING, job_key STRING, is_applied BOOLEAN, applied_at STRING, is_hidden BOOLEAN, hidden_at STRING, "
                "application_status STRING, status_updated_at STRING")
CUSTOM_ROLE_SCHEMA = ("role_title STRING, status STRING, category STRING, description STRING, duplicate_of STRING, "
                      "first_user STRING, checked_by STRING, created_at STRING, checked_at STRING, attempts INT")
CUSTOM_SKILL_SCHEMA = "skill STRING, first_user STRING, created_at STRING"


class UserDataError(ValueError):
    pass


def utc_timestamp(value, field):
    """ISO 8601 (Z / offset, any fraction) -> 'YYYY-MM-DDTHH:MM:SS.ffffffZ' in UTC; None stays None."""
    if value is None:
        return None
    match = ISO_TS_RE.fullmatch(value.strip()) if isinstance(value, str) else None
    if not match:
        raise UserDataError(f"{field} is not an ISO timestamp")
    try:
        parsed = datetime.strptime(f"{match.group(1)}T{match.group(2)}", "%Y-%m-%dT%H:%M:%S")
    except ValueError:
        raise UserDataError(f"{field} is not a valid date and time") from None
    parsed = parsed.replace(microsecond=int((match.group(3) or "0").ljust(6, "0")[:6]), tzinfo=timezone.utc)
    zone = match.group(4) or "Z"
    if zone != "Z":
        digits = zone[1:].replace(":", "")
        offset = timedelta(hours=int(digits[:2]), minutes=int(digits[2:]))
        parsed = parsed - offset if zone[0] == "+" else parsed + offset
    return parsed.strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def text(value, field, required=False):
    if value is None and not required:
        return None
    if not isinstance(value, str) or (required and not value.strip()):
        raise UserDataError(f"{field} must be a{' non-empty' if required else ''} string")
    return value


def integer(value, field):
    if value is None:
        return None
    if isinstance(value, bool) or not isinstance(value, int) or not -2**31 <= value < 2**31:
        raise UserDataError(f"{field} must be an integer")
    return value


def boolean(value, field):
    if value is None:
        return False
    if not isinstance(value, bool):
        raise UserDataError(f"{field} must be true or false")
    return value


def string_list(value, field):
    if value is None:
        return []
    if not isinstance(value, list) or not all(isinstance(item, str) for item in value):
        raise UserDataError(f"{field} must be a list of strings")
    return value


def unique(rows, key, label):
    seen = set()
    for row in rows:
        if key(row) in seen:
            raise UserDataError(f"duplicate {label}")
        seen.add(key(row))
    return rows


def uuid_text(value, field):
    value = text(value, field, required=True).lower()
    if not UUID_RE.fullmatch(value):
        raise UserDataError(f"{field} is not a uuid")
    return value


def job_key_text(value):
    if not isinstance(value, str) or not JOB_KEY_RE.fullmatch(value):
        raise UserDataError("job_state.job_key is not a job key")
    return value


def role_status(value):
    if value not in CUSTOM_ROLE_STATUSES:
        raise UserDataError("custom_roles.status is not pending / active / mapped / rejected")
    return value


def parse_user_data(raw):
    """user_data.json bytes -> (exported_at, {name: rows as tuples in the *_SCHEMA order}); raises UserDataError."""
    try:
        doc = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise UserDataError(f"not valid JSON ({type(exc).__name__})") from None
    if not isinstance(doc, dict) or doc.get("format") != USER_DATA_FORMAT:
        raise UserDataError(f"format is not {USER_DATA_FORMAT}")
    if not doc.get("exported_at"):
        raise UserDataError("exported_at is missing")
    exported_at = utc_timestamp(doc["exported_at"], "exported_at")
    counts = doc.get("counts") if isinstance(doc.get("counts"), dict) else {}
    for name in ("profiles", "job_state", "custom_roles", "custom_skills"):
        if not isinstance(doc.get(name), list) or not all(isinstance(row, dict) for row in doc[name]):
            raise UserDataError(f"{name} is not a list of objects")
        if counts.get(name) != len(doc[name]):
            raise UserDataError(f"counts.{name} does not match its list")

    profiles = unique([(
        uuid_text(p.get("profile_id"), "profiles.profile_id"),
        text(p.get("email"), "profiles.email"),
        string_list(p.get("target_roles"), "profiles.target_roles"),
        string_list(p.get("skills"), "profiles.skills"),
        integer(p.get("min_years"), "profiles.min_years"),
        integer(p.get("max_years"), "profiles.max_years"),
        string_list(p.get("preferred_cities"), "profiles.preferred_cities"),
        utc_timestamp(p.get("updated_at"), "profiles.updated_at"),
    ) for p in doc["profiles"]], lambda row: row[0], "profile_id")

    job_state = unique([(
        uuid_text(s.get("user_id"), "job_state.user_id"),
        job_key_text(s.get("job_key")),
        boolean(s.get("is_applied"), "job_state.is_applied"),
        utc_timestamp(s.get("applied_at"), "job_state.applied_at"),
        boolean(s.get("is_hidden"), "job_state.is_hidden"),
        utc_timestamp(s.get("hidden_at"), "job_state.hidden_at"),
        text(s.get("application_status"), "job_state.application_status") or "not_applied",
        utc_timestamp(s.get("status_updated_at"), "job_state.status_updated_at"),
    ) for s in doc["job_state"]], lambda row: (row[0], row[1]), "job state row")

    custom_roles = unique([(
        text(r.get("role_title"), "custom_roles.role_title", required=True),
        role_status(r.get("status")),
        text(r.get("category"), "custom_roles.category"),
        text(r.get("description"), "custom_roles.description"),
        text(r.get("duplicate_of"), "custom_roles.duplicate_of"),
        text(r.get("first_user"), "custom_roles.first_user"),
        text(r.get("checked_by"), "custom_roles.checked_by"),
        utc_timestamp(r.get("created_at"), "custom_roles.created_at"),
        utc_timestamp(r.get("checked_at"), "custom_roles.checked_at"),
        integer(r.get("attempts"), "custom_roles.attempts"),
    ) for r in doc["custom_roles"]], lambda row: row[0].lower(), "custom role")

    custom_skills = unique([(
        text(s.get("skill"), "custom_skills.skill", required=True),
        text(s.get("first_user"), "custom_skills.first_user"),
        utc_timestamp(s.get("created_at"), "custom_skills.created_at"),
    ) for s in doc["custom_skills"]], lambda row: row[0].lower(), "custom skill")

    return exported_at, {"profiles": profiles, "job_state": job_state, "custom_roles": custom_roles,
                         "custom_skills": custom_skills}


def written(result, key="num_affected_rows"):
    row = result.first()
    return int(row[key] or 0) if row is not None and key in row.asDict() else 0


def overwrite_select(target, values):
    """SELECT list in the target's column order: the given columns cast to the column type, NULL for the others."""
    fields = spark.table(target).schema.fields
    unknown = set(values) - {field.name for field in fields}
    if unknown:
        raise RuntimeError(f"{target} has no columns {sorted(unknown)}; run 01_setup")
    return ", ".join(
        f"CAST({values.get(field.name, 'NULL')} AS {field.dataType.simpleString()}) AS {field.name}" for field in fields
    )


def load_user_data(metrics):
    details = metrics["details"]
    details.update(loaded=False, file=USER_DATA_FILE)
    metrics.update(rows_in=0, rows_out=0, rows_rejected=0)
    try:
        with open(USER_DATA_FILE, "rb") as handle:
            raw = handle.read()
    except FileNotFoundError:
        details["skipped_reason"] = "missing"
        metrics["message"] = "no user_data.json; user tables kept"
        print(f"{USER_DATA_FILE} not found: user tables kept")
        return
    except OSError as exc:
        details["skipped_reason"] = f"unreadable: {type(exc).__name__}"
        metrics["message"] = "user_data.json unreadable; user tables kept"
        print(f"[WARN] {USER_DATA_FILE} unreadable ({type(exc).__name__}): user tables kept")
        return
    details["bytes"] = len(raw)

    try:
        exported_at, rows = parse_user_data(raw)
    except UserDataError as exc:
        details["skipped_reason"] = f"invalid: {exc}"[:300]
        metrics["message"] = "invalid user_data.json; user tables kept"
        print(f"[WARN] {USER_DATA_FILE} is invalid ({exc}): user tables kept")
        return
    details["exported_at"] = exported_at
    details["file_counts"] = {name: len(value) for name, value in rows.items()}
    metrics["rows_in"] = sum(details["file_counts"].values())

    # exported_at is stored normalised (UTC, 6 fraction digits), so the string max is the latest load
    last_loaded = spark.sql(f"""
        SELECT max(get_json_object(details, '$.exported_at')) AS exported_at FROM {RUNS_TABLE}
        WHERE task = 'load_user_data' AND status = 'SUCCESS' AND get_json_object(details, '$.loaded') = 'true'
    """).first()["exported_at"]
    if last_loaded and exported_at <= last_loaded:
        details.update(skipped_reason="unchanged", last_loaded_exported_at=last_loaded)
        metrics["message"] = "user_data.json not newer than the last load"
        print(f"user_data.json exported_at={exported_at} already loaded (last load {last_loaded}): skipped")
        return

    existing_profiles = int(spark.sql(f"SELECT count(*) AS n FROM {PROFILE_TABLE} WHERE profile_id <> 'default'").first()["n"])
    if not rows["profiles"] and existing_profiles:
        details.update(skipped_reason="empty export", existing_profiles=existing_profiles)
        metrics["message"] = "empty export while ops.user_profile has profiles; user tables kept"
        print(f"[WARN] user_data.json has no profiles but ops.user_profile has {existing_profiles}: nothing loaded")
        return

    spark.createDataFrame(rows["profiles"], PROFILE_SCHEMA).createOrReplaceTempView("user_data_profiles")
    spark.createDataFrame(rows["job_state"], STATE_SCHEMA).createOrReplaceTempView("user_data_job_state")
    spark.createDataFrame(rows["custom_roles"], CUSTOM_ROLE_SCHEMA).selectExpr(
        "role_title", "status", "category", "description", "duplicate_of", "first_user", "checked_by",
        "CAST(created_at AS TIMESTAMP) AS created_at", "CAST(checked_at AS TIMESTAMP) AS checked_at", "attempts",
    ).createOrReplaceTempView("user_data_custom_roles")
    spark.createDataFrame(rows["custom_skills"], CUSTOM_SKILL_SCHEMA).selectExpr(
        "skill", "first_user", "CAST(created_at AS TIMESTAMP) AS created_at",
    ).createOrReplaceTempView("user_data_custom_skills")

    # Timestamps arrive as '...Z' strings: CAST(... AS TIMESTAMP) honours the zone, whatever the session time zone is
    timings = details["seconds"] = {}
    started = time.monotonic()
    profile_columns = ["profile_id", "email", "target_roles", "skills", "min_years", "max_years", "preferred_cities",
                       "updated_at"]
    profiles_written = written(spark.sql(f"""
        INSERT OVERWRITE {PROFILE_TABLE}
        SELECT {overwrite_select(PROFILE_TABLE, {column: column for column in profile_columns})}
        FROM user_data_profiles
    """))
    timings["profiles"] = round(time.monotonic() - started, 1)

    started = time.monotonic()
    state_columns = ["user_id", "job_key", "is_applied", "applied_at", "is_hidden", "hidden_at", "application_status",
                     "status_updated_at"]
    state_written = written(spark.sql(f"""
        INSERT OVERWRITE {USER_STATE_TABLE}
        SELECT {overwrite_select(USER_STATE_TABLE, {column: column for column in state_columns})}
        FROM user_data_job_state
    """))
    timings["job_state"] = round(time.monotonic() - started, 1)

    # Custom roles: 03_enrich checks them; the newer check wins and nothing is deleted
    started = time.monotonic()
    roles_result = spark.sql(f"""
        MERGE INTO {CUSTOM_ROLES_TABLE} t
        USING user_data_custom_roles s
        ON lower(t.role_title) = lower(s.role_title)
        WHEN MATCHED AND (t.checked_at IS NULL OR s.checked_at > t.checked_at) THEN UPDATE SET
          t.status = s.status, t.category = s.category, t.description = s.description, t.duplicate_of = s.duplicate_of,
          t.checked_by = s.checked_by, t.checked_at = s.checked_at, t.attempts = greatest(t.attempts, s.attempts),
          t.first_user = coalesce(t.first_user, s.first_user), t.created_at = coalesce(t.created_at, s.created_at)
        WHEN NOT MATCHED THEN INSERT (role_title, status, category, description, duplicate_of, first_user, checked_by,
                                      created_at, checked_at, attempts)
          VALUES (s.role_title, s.status, s.category, s.description, s.duplicate_of, s.first_user, s.checked_by,
                  s.created_at, s.checked_at, s.attempts)
    """)
    skills_result = spark.sql(f"""
        MERGE INTO {CUSTOM_SKILLS_TABLE} t
        USING user_data_custom_skills s
        ON lower(t.skill) = lower(s.skill)
        WHEN NOT MATCHED THEN INSERT (skill, first_user, created_at) VALUES (s.skill, s.first_user, s.created_at)
    """)
    timings["custom"] = round(time.monotonic() - started, 1)

    counts = {
        "profiles": profiles_written,
        "job_state": state_written,
        "custom_roles_inserted": written(roles_result, "num_inserted_rows"),
        "custom_roles_updated": written(roles_result, "num_updated_rows"),
        "custom_skills_inserted": written(skills_result, "num_inserted_rows"),
    }
    details.update(loaded=True, counts=counts, replaced_profiles=existing_profiles)
    metrics["rows_out"] = sum(counts.values())
    print(f"user_data.json exported_at={exported_at} loaded: {counts} (replaced {existing_profiles} profiles) {timings}")


# A problem here must never stop the scraped data: task_run logs FAILED and the run carries on with the tables as they
# are. A partly applied load is redone by the next run (it finds no successful load of this export).
try:
    with task_run("load_user_data") as metrics:
        load_user_data(metrics)
except Exception as exc:
    print(f"[WARN] load_user_data failed; the next run loads user_data.json again: {type(exc).__name__}: "
          f"{(str(exc).splitlines() or [''])[0][:300]}")

## 1. Stream new files into bronze

In [ ]:
with task_run("bronze_ingest") as metrics:
    raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("pathGlobFilter", "*.csv")
        .option("rescuedDataColumn", "_rescued_data")
        .schema(CSV_SCHEMA)
        .load(LANDING_PATH)
    )

    row_hash = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(column), F.lit("")) for column in BUSINESS_COLUMNS]), 256)
    landing_folder_date = F.expr(
        "CAST(try_to_timestamp(regexp_extract(_metadata.file_path, '/([0-9]{8})/[^/]+$', 1), 'ddMMyyyy') AS DATE)"
    )

    bronze = raw.select(
        *BUSINESS_COLUMNS,
        F.col("_rescued_data"),
        F.col("_metadata.file_path").alias("_source_file"),
        F.lower(F.split(F.col("_metadata.file_name"), "_").getItem(0)).alias("_source_system"),
        landing_folder_date.alias("_landing_folder_date"),
        F.col("_metadata.file_modification_time").alias("_file_modification_time"),
        F.col("_metadata.file_size").alias("_file_size"),
        row_hash.alias("_row_hash"),
        F.current_timestamp().alias("_ingested_at"),
        F.lit(JOB_RUN_ID).alias("_job_run_id"),
    )

    query = (
        bronze.writeStream
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
    )
    query.awaitTermination()

    # One action for all bronze counters (each file belongs to exactly one source system)
    by_source = (
        spark.table(BRONZE_TABLE).where(F.col("_job_run_id") == JOB_RUN_ID)
        .groupBy("_source_system")
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct("_source_file").alias("files"),
            F.sum(F.col("_rescued_data").isNotNull().cast("int")).alias("rescued_rows"),
        )
        .collect()
    )
    rows = sum(int(row["rows"]) for row in by_source)
    rescued_rows = sum(int(row["rescued_rows"] or 0) for row in by_source)

    metrics.update(
        files_read=sum(int(row["files"]) for row in by_source),
        rows_in=rows,
        rows_out=rows,
        rows_rejected=rescued_rows,
        details={"by_source": {row["_source_system"]: int(row["rows"]) for row in by_source}, "rescued_rows": rescued_rows},
        message="no new files" if rows == 0 else None,
    )

## 2. Find unprocessed bronze runs

In [ ]:
if FULL_REFRESH in ("silver", "all"):
    processed_runs = spark.createDataFrame([], "run_id STRING")
else:
    processed_runs = spark.sql(f"""
        SELECT explode(from_json(details, 'struct<bronze_run_ids:array<string>>').bronze_run_ids) AS run_id
        FROM {RUNS_TABLE}
        WHERE task = 'silver_transform' AND status = 'SUCCESS'
    """)

pending_run_ids = sorted(
    row["_job_run_id"]
    for row in spark.table(BRONZE_TABLE).select("_job_run_id").distinct()
    .join(processed_runs, F.col("_job_run_id") == F.col("run_id"), "left_anti")
    .collect()
)
print(f"Pending bronze runs: {pending_run_ids or 'none'}")

## Cleaning helpers

In [ ]:
def clean_string(column):
    trimmed = F.trim(F.col(column))
    return F.when(trimmed == "", None).otherwise(trimmed)


def parse_dmy(column):
    return F.expr(f"CAST(try_to_timestamp({column}, 'dd/MM/yyyy') AS DATE)")


def dml_counts(result):
    """Row counts reported by a Delta INSERT / MERGE statement (saves separate count() actions)."""
    row = result.first()
    return {key: int(value or 0) for key, value in row.asDict().items()} if row else {}


# --- LEGACY CSV FORMAT: remove after 2026-10-13, when old LinkedIn files have aged out of the landing volume ---
# Old linkedin.py files prefix company names with 'LinkedIn | ' ('LinkedIn' alone = company unknown) and append the job
# criteria to the description instead of filling the criteria columns. Only rows that look like that go through the
# Python UDF below; every other row is cleaned with native Spark expressions.
CRITERIA_LABELS = ["Seniority level", "Employment type", "Job function", "Industries"]
LABEL_PATTERN = "|".join(re.escape(label) for label in CRITERIA_LABELS)
TRAILING_LABELS_RE = re.compile(rf"((?:{LABEL_PATTERN})(?: (?:{LABEL_PATTERN}))*)\s*$")

criteria_schema = StructType([
    StructField("description", StringType()),
    *[StructField(column, StringType()) for column in CRITERIA_COLUMNS],
])


@F.udf(returnType=criteria_schema)
def split_linkedin_criteria(text):
    """LinkedIn descriptions end with 'L1 V1 L2 V2 ... V1 V2 ... L1 L2 ...'. Extract V1..Vn and drop the block."""
    if not text:
        return (text, None, None, None, None)
    trailing = TRAILING_LABELS_RE.search(text)
    if not trailing:
        return (text, None, None, None, None)

    labels = re.findall(LABEL_PATTERN, trailing.group(1))
    pattern = (
        " (.+?) ".join(re.escape(label) for label in labels)
        + " (.+?) "
        + " ".join(rf"\{index + 1}" for index in range(len(labels)))
        + r"\s+" + re.escape(trailing.group(1)) + r"\s*$"
    )
    block = re.search(pattern, text)
    if not block:
        return (text[:trailing.start()].strip() or None, None, None, None, None)

    values = dict(zip(labels, (value.strip() for value in block.groups())))
    return (
        text[:block.start()].strip() or None,
        values.get("Seniority level"),
        values.get("Employment type"),
        values.get("Job function"),
        values.get("Industries"),
    )


def is_legacy_linkedin_row():
    """LinkedIn row without criteria columns whose description ends with the criteria labels (old file format)."""
    no_criteria = F.coalesce(*[F.col(column) for column in CRITERIA_COLUMNS]).isNull()
    trailing_labels = F.col("description_raw").rlike(rf"({LABEL_PATTERN})\s*$")
    return F.coalesce((F.col("ats_type") == "linkedin") & no_criteria & trailing_labels, F.lit(False))


def is_legacy_linkedin_file():
    """Every row of a LinkedIn file in which no row has a criteria value (old file format; a new file whose jobs were all
    already seen looks the same, which only matters for a company literally named 'LinkedIn')."""
    has_criteria = F.coalesce(*[F.col(column) for column in CRITERIA_COLUMNS]).isNotNull().cast("int")
    no_criteria_in_file = F.max(has_criteria).over(Window.partitionBy("_source_file")) == 0
    return F.coalesce((F.col("ats_type") == "linkedin") & no_criteria_in_file, F.lit(False))


def without_legacy_company_prefix(column):
    company = F.regexp_replace(F.col(column), r"^LinkedIn\s*\|\s*", "")
    return F.when(F.col("_legacy_file"), F.regexp_replace(company, r"^LinkedIn$", "")).otherwise(company)
# --- END LEGACY CSV FORMAT ---

## 3. Clean and MERGE into silver

In [ ]:
with task_run("silver_transform") as metrics:
    metrics["details"]["bronze_run_ids"] = pending_run_ids

    if not pending_run_ids:
        metrics.update(rows_in=0, rows_out=0, rows_rejected=0, message="nothing to process")
    else:
        bronze = spark.table(BRONZE_TABLE).where(F.col("_job_run_id").isin(pending_run_ids))

        base = bronze.select(
            *[clean_string(column).alias(column) for column in [
                "company_name", "company_url", "job_id", "title", "location", "job_url", "posted_date", "collected_on",
                "posted_days", *CRITERIA_COLUMNS,
            ]],
            F.lower(clean_string("ats_type")).alias("ats_type"),
            F.regexp_replace(F.col("description"), r"\s+", " ").alias("description_raw"),
            "_rescued_data", "_source_file", "_source_system", "_file_modification_time", "_row_hash", "_ingested_at",
        )

        reasons = F.concat_ws(",", F.array(
            F.when(F.col("job_id").isNull(), F.lit("missing_job_id")),
            F.when(F.col("title").isNull(), F.lit("missing_title")),
            F.when(F.col("ats_type").isNull(), F.lit("missing_ats_type")),
            F.when(F.col("job_url").isNull(), F.lit("missing_job_url")),
        ))
        base = base.withColumn("reject_reason", F.when(reasons != "", reasons))
        # linkedin.py writes 'Not specified' for cards without a location: unknown, so a stored location is kept
        base = base.withColumn("location", F.when(F.lower("location") == "not specified", None).otherwise(F.col("location")))

        # INSERT reports its row count, so the rejected rows are not counted separately
        base.where(F.col("reject_reason").isNotNull()).createOrReplaceTempView("silver_rejected_rows")
        rows_rejected = dml_counts(spark.sql(f"""
            INSERT INTO {REJECTED_TABLE} ({", ".join(REJECTED_COLUMNS)})
            SELECT {", ".join(REJECTED_COLUMNS[:-2])}, '{JOB_RUN_ID}', current_timestamp()
            FROM silver_rejected_rows
        """)).get("num_inserted_rows", 0)

        valid = (
            base.where(F.col("reject_reason").isNull())
            .withColumn("_legacy", is_legacy_linkedin_row())
            .withColumn("_legacy_file", is_legacy_linkedin_file())
        )
        # Legacy rows (criteria inside the description) need the UDF; filtering first keeps it off all other rows
        current_rows = valid.where(~F.col("_legacy")).withColumn("description", clean_string("description_raw"))
        legacy_rows = (
            valid.where(F.col("_legacy"))
            .withColumn("_criteria", split_linkedin_criteria(F.col("description_raw")))
            .withColumns({column: F.col(f"_criteria.{column}") for column in ["description", *CRITERIA_COLUMNS]})
            .drop("_criteria")
        )

        is_linkedin = F.col("ats_type") == "linkedin"
        company = F.trim(without_legacy_company_prefix("company_name"))
        valid = (
            current_rows.unionByName(legacy_rows)
            .withColumn("company_name", F.when((company == "") | company.isNull(), F.lit("Unknown")).otherwise(company))
            .withColumn("job_id", F.when(is_linkedin & F.col("job_id").rlike("[0-9]+$"), F.regexp_extract("job_id", r"([0-9]+)$", 1)).otherwise(F.col("job_id")))
            .withColumn("job_url", F.when(is_linkedin, F.split("job_url", r"\?").getItem(0)).otherwise(F.col("job_url")))
            .withColumn("job_key", F.sha2(F.concat_ws("|", F.col("ats_type"), F.lower(F.col("company_name")), F.col("job_id")), 256))
            .withColumn("_seen_at", F.coalesce(F.col("_file_modification_time"), F.col("_ingested_at")))
        )

        # Dedupe within this batch: the latest sighting that has a description wins; keep sighting stats
        latest_first = Window.partitionBy("job_key").orderBy(
            F.col("description").isNull(), F.col("_seen_at").desc(), F.col("_ingested_at").desc(), F.col("_row_hash"),
        )
        sightings = valid.groupBy("job_key").agg(
            F.min("_seen_at").alias("first_seen_at"),
            F.max("_seen_at").alias("last_seen_at"),
            F.countDistinct("_source_file").cast("int").alias("times_seen"),
            F.count("*").alias("_rows"),
            F.sum(F.col("_legacy").cast("int")).alias("_legacy_rows"),
        )
        latest = valid.withColumn("_rank", F.row_number().over(latest_first)).where("_rank = 1").join(sightings, "job_key")

        # Location -> is_india using the reference aliases (word-boundary match), on the distinct locations only
        india_aliases = F.broadcast(spark.table(INDIA_REF_TABLE).select(F.lower("alias").alias("alias")).distinct())
        india_locations = (
            latest.select("location").where(F.col("location").isNotNull()).distinct()
            .crossJoin(india_aliases)
            .where(F.expr("lower(location) RLIKE concat('(^|[^a-z])', alias, '([^a-z]|$)')"))
            .select("location").distinct().withColumn("is_india", F.lit(True))
        )
        first_location = F.trim(F.split(F.col("location"), ";").getItem(0))
        last_part = F.trim(F.element_at(F.split(first_location, ","), -1))

        updates = (
            latest.join(india_locations, "location", "left")
            .withColumn("is_india", F.coalesce(F.col("is_india"), F.lit(False)))
            .withColumn("country", F.when(F.col("is_india"), F.lit("India"))
                        .when(F.col("location").isNull() | F.col("location").rlike(r"(?i)^[0-9]+ locations?$"), F.lit(None))
                        .otherwise(last_part))
            .select(
                "job_key",
                F.coalesce(F.col("ats_type"), F.col("_source_system")).alias("source"),
                "ats_type", "company_name", "company_url", "job_id", "title", "location", "country", "is_india",
                parse_dmy("posted_date").alias("posted_date"),
                F.expr("CAST(TRY_CAST(posted_days AS DOUBLE) AS INT)").alias("posted_days"),
                parse_dmy("collected_on").alias("collected_on"),
                "job_url", "description",
                F.length("description_raw").alias("description_raw_length"),
                F.sha2(F.coalesce(F.col("description"), F.lit("")), 256).alias("description_hash"),
                *CRITERIA_COLUMNS,
                (F.col("description").isNull() | (F.length("description") < 50)).alias("dq_missing_description"),
                "first_seen_at", "last_seen_at", "times_seen", "_source_file", "_row_hash", "_ingested_at",
                F.lit(JOB_RUN_ID).alias("_job_run_id"),
                F.current_timestamp().alias("_updated_at"),
                "_rows", "_legacy_rows",
            )
        )

        # One action for the batch counters; non-India jobs are only counted
        stats = updates.agg(
            F.sum("_rows").alias("rows"),
            F.sum("_legacy_rows").alias("legacy_rows"),
            F.count("*").alias("jobs"),
            F.sum(F.col("is_india").cast("int")).alias("india_jobs"),
            F.sum(F.col("location").isNull().cast("int")).alias("no_location_jobs"),
        ).first()

        # Jobs without a location may still refresh a stored (India) job, but are never inserted
        updates.where("is_india OR location IS NULL").createOrReplaceTempView("silver_updates")

        # A re-sighting without a value never overwrites a stored one: LinkedIn re-emits jobs already in silver
        # (config linkedin_seen_ids.json) without their description. Derived columns follow their source column.
        keep_if_empty = {*CRITERIA_COLUMNS, "posted_date", "posted_days"}
        follows = {
            "location": "location", "country": "location", "is_india": "location",
            "description": "description", "description_raw_length": "description",
            "description_hash": "description", "dq_missing_description": "description",
        }

        def refresh(column):
            if column in keep_if_empty:
                return f"t.{column} = coalesce(s.{column}, t.{column})"
            if column in follows:
                return f"t.{column} = CASE WHEN s.{follows[column]} IS NULL THEN t.{column} ELSE s.{column} END"
            return f"t.{column} = s.{column}"

        refresh_columns = [c for c in SILVER_COLUMNS if c not in ("job_key", "first_seen_at", "last_seen_at", "times_seen")]
        full_update = ",\n              ".join(refresh(c) for c in refresh_columns)
        merged = dml_counts(spark.sql(f"""
            MERGE INTO {SILVER_TABLE} t
            USING silver_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED AND s.last_seen_at >= t.last_seen_at THEN UPDATE SET
              {full_update},
              t.first_seen_at = least(t.first_seen_at, s.first_seen_at),
              t.last_seen_at = s.last_seen_at,
              t.times_seen = t.times_seen + s.times_seen
            WHEN MATCHED THEN UPDATE SET
              t.first_seen_at = least(t.first_seen_at, s.first_seen_at),
              t.times_seen = t.times_seen + s.times_seen,
              t._job_run_id = s._job_run_id,
              t._updated_at = s._updated_at
            WHEN NOT MATCHED AND s.is_india THEN INSERT ({", ".join(SILVER_COLUMNS)})
              VALUES ({", ".join(f"s.{c}" for c in SILVER_COLUMNS)})
        """))

        jobs = int(stats["jobs"])
        india_jobs, no_location_jobs = int(stats["india_jobs"] or 0), int(stats["no_location_jobs"] or 0)
        metrics.update(
            rows_in=int(stats["rows"] or 0) + rows_rejected,
            rows_out=merged.get("num_affected_rows", 0),
            rows_rejected=rows_rejected,
        )
        metrics["details"].update(
            unique_jobs_in_batch=jobs,
            india_jobs=india_jobs,
            non_india_jobs_skipped=jobs - india_jobs - no_location_jobs,
            no_location_jobs=no_location_jobs,
            legacy_format_rows=int(stats["legacy_rows"] or 0),
            inserted=merged.get("num_inserted_rows", 0),
            updated=merged.get("num_updated_rows", 0),
        )

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect this run

In [ ]:
display(
    spark.table(BRONZE_TABLE)
    .where(F.col("_job_run_id") == JOB_RUN_ID)
    .groupBy("_source_system", "_landing_folder_date", "_source_file")
    .agg(F.count("*").alias("rows"), F.sum(F.when(F.col("_rescued_data").isNotNull(), 1).otherwise(0)).alias("rescued_rows"))
    .orderBy("_landing_folder_date", "_source_system", "_source_file")
)

In [ ]:
silver = spark.table(SILVER_TABLE)
display(
    silver.groupBy("source")
    .agg(
        F.count("*").alias("jobs"),
        F.sum(F.col("is_india").cast("int")).alias("india_jobs"),
        F.sum(F.col("dq_missing_description").cast("int")).alias("missing_description"),
        F.sum(F.col("posted_date").isNull().cast("int")).alias("missing_posted_date"),
        F.sum(F.col("seniority_level").isNotNull().cast("int")).alias("with_linkedin_criteria"),
        F.round(F.avg("times_seen"), 2).alias("avg_times_seen"),
    )
    .orderBy("source")
)

In [ ]:
display(
    silver.where("source = 'linkedin'")
    .select("company_name", "job_id", "title", "seniority_level", "employment_type", "job_function", "industries",
            F.substring("description", -120, 120).alias("description_tail"))
    .limit(10)
)

In [ ]:
display(silver.groupBy("source", "country").count().orderBy(F.col("count").desc()).limit(25))

In [ ]:
display(spark.table(REJECTED_TABLE).groupBy("reject_reason", "ats_type").count().orderBy(F.col("count").desc()))

In [ ]:
display(
    spark.table(RUNS_TABLE).where("task IN ('load_user_data', 'bronze_ingest', 'silver_transform')")
    .orderBy(F.col("started_at").desc()).limit(10)
)